# `eval` — the orchestrator, end to end

The other notebook (`HPT_a100_ui_eval`) drives the four prompts by hand. This one
runs the thing that actually ships: `HealthcarePricingPipeline` — the same code path
a user's question takes.

| stage | what runs here |
|---|---|
| retrieval | `CodeRetriever` over the Qdrant collections, `cpt_medte` and `hcpcs_medte` — the app's own store |
| generation | MedGemma via vLLM, **in this process**, one set of weights |
| orchestration | entity extraction → retrieval → code decision → validation → price lookup → answer |

Each of the 600 eval queries is run, and the result is written **beside** the key in
new columns. The key is copied through untouched — nothing here edits the dataset.

| generated column | metric it feeds |
|---|---|
| `generate_cpt`, `generate_hcpcs` | Sorensen-Dice, Jaccard vs the key's codes; pivot hit |
| `generate_row_ids` | Dice / Jaccard over MRF row ids, SQL row-set exactness, by-stratum |
| `generate_rate`, `generate_cash` | rate citation fidelity (½¢ after rounding), right-number-wrong-row |
| `generate_hospital`, `generate_insurer`, `generate_medical`, `generate_medication` | precision / recall / F1 per field, then macro |
| `generate_intent` | accuracy, per-class F1 |
| `generate_answer` | abstention: correct silence vs false price |
| `generate_error`, `generate_seconds` | parse failures, latency |

## 1. Code

In [ ]:
%cd /content
import os
if not os.path.isdir("/content/HPT_Prototype"):
    !git clone -q https://github.com/adi274903/HPT_Prototype /content/HPT_Prototype
%cd /content/HPT_Prototype

# the orchestrator eval lives on its own branch, cut from no-router
!git fetch -q origin orchestrator-eval
!git checkout -q -B orchestrator-eval FETCH_HEAD
!git log --oneline -1                     # 0e42f72 or newer
!pip install -q duckdb

from pt_healthcare.eval_pipeline import GENERATED_COLUMNS, generate, score
print("eval_pipeline ready:", len(GENERATED_COLUMNS), "generated columns")

## 2. Drive and the Qdrant store

The store is the ~2 GB unpacked snapshot on Drive; `stage_db` copies it to
`/content/New_PT_DB` once. The eval queries it in place — same collections the app
reads, so there is nothing to build and no catalog CSV to stage.

In [ ]:
import colab_run

colab_run.mount_drive()
qdrant_path = colab_run.stage_db()
print(colab_run.default_paths())

## 3. One pipeline — the app's own

`setup()` wires `CodeRetriever` over that store and `MedGemmaEngine` over vLLM in
this process. Everything below reuses this object: one Qdrant client, so no
"already accessed by another instance" lock, and one copy of the weights.

In [ ]:
pipeline = colab_run.setup(mount=False, stage=False, verbose=True)

# the orchestrator narrates seven steps per case; 600 cases is not a place for that
pipeline.printer = lambda _message: None
pipeline.display_fn = lambda *_args, **_kwargs: None

print("retriever:", type(pipeline.retriever).__name__, "| top_k:", pipeline.top_k)
print("engine:   ", type(pipeline.engine).__name__)
print("MRF rows: ", len(pipeline.mrf_data))

## 4. Smoke — ten cases, one per subtype

Cheap enough to catch a wiring problem before 600 cases. Watch for three things:
non-empty `generate_cpt`, row ids that come back as ids and not strings of a list,
and a `generate_seconds` that looks like real inference.

In [ ]:
rows = generate(pipeline=pipeline, dataset="eval",
                out="/content/eval_smoke.jsonl", per_subtype=1, fresh=True)

first = rows[0]
for column in GENERATED_COLUMNS:
    print(f"{column:26} {str(first[column])[:88]}")

## 5. The full 600

The run appends every finished case to `/content/eval_generations.jsonl` and skips
what is already there, so a dropped runtime costs the case in flight — re-run the
cell and it continues. `fresh=True` starts over.

`with_intent=True` adds one routing call per case for the intent column. This branch
has no router stage; without the flag the report says *not measured* rather than
printing a zero.

In [ ]:
rows = generate(pipeline=pipeline, dataset="eval",
                out="/content/eval_generations.jsonl",
                csv_out="/content/eval_generations.csv",
                with_intent=True)
print(len(rows), "cases on disk")

## 6. Score the columns

No model and no Qdrant here — it judges what is already on disk, so a metric tweak
costs seconds instead of another 600-case run. `mrf_data=pipeline.mrf_data` prices
the key's rows off the frame already in memory (pass `--db` / `db=` to use the
duckdb file instead).

In [ ]:
report = score(rows="/content/eval_generations.jsonl", dataset="eval",
               mrf_data=pipeline.mrf_data,
               report_path="/content/eval_report.json",
               scores_out="/content/eval_scores.jsonl")

from pt_healthcare.eval_runner import format_report
print(format_report(report))

## 7. The table, and where the numbers came from

In [ ]:
import pandas as pd

generations = pd.read_csv("/content/eval_generations.csv")
print(generations.shape)
print(generations[["eval_id", "generate_cpt", "generate_row_ids",
                   "generate_rate", "generate_seconds"]].head(10).to_string(index=False))

scores = pd.read_json("/content/eval_scores.jsonl", lines=True)
scores["dice"] = scores["codes"].apply(lambda value: value.get("dice_mrf"))
scores["rows_dice"] = scores["rows"].apply(lambda value: value.get("dice_raw"))
print(scores.groupby("subtype")[["dice", "rows_dice"]].mean().round(3))

## Notes

**What the numbers mean.** `dice_mrf` is scored against the codes that exist as MRF
rows for that case, so a run that returns the right service among five codes scores
badly on Dice by construction — read `pivot_hit` alongside it. `sql_exact` is the
negotiated-rate check: the returned row set has to match, not just the figure.

**Running it without the UI.** The same two phases exist as a CLI, for a fresh
process where nothing holds the store:

```python
!python -m pt_healthcare.eval_pipeline generate --qdrant-path /content/New_PT_DB \
    --mrf "/content/drive/MyDrive/.../everyUPMCmrf_clean (1).csv" \
    --out /content/eval_generations.jsonl --with-intent
!python -m pt_healthcare.eval_pipeline score --rows /content/eval_generations.jsonl \
    --db .cache/mrf_eval.duckdb --report /content/eval_report.json
```

That path builds its own Qdrant client, so free the lock first — `ui.stop()`,
`pipeline.retriever.client.close()`, then the `gc.get_objects()` sweep of
`QdrantClient` in the reset cell.

**Cost.** 600 cases × roughly five model calls each. The smoke run's
`generate_seconds` × 600 is a fair estimate before you commit the GPU.

**Narrowing a sweep.** `subtypes=["C1", "D1"], per_subtype=20, out="/content/slice.jsonl"`
— separate `out` per slice, or the resume logic will skip cases you meant to rerun.